# Stage 2: Train & Evaluate Aerial Vehicle Detector (`02_train_and_evaluate_detector.ipynb`)

### End-to-End Deep Learning Pipeline for thefourthlaw.ai Test Task

This notebook trains and evaluates a lightweight neural detector under strict real-world constraints:
- **Rule 1 (Data Split Isolation)**: Video-level isolation. `eval` (`32179597.mp4`) and `eval_B` (`13763017.mp4`) are strictly held-out in `test` split. No test frames are seen during training or threshold calibration.
- **Rule 4 (Optical Distance Bands)**: Evaluated across both `0–200 m` and `200–400 m` using pinhole optics and physical vehicle dimensions (4.5 m length, 1.8 m width).
- **Rule 5 (Edge Model Constraints)**: Custom **YOLO26-Pico-P2** architecture (< 500,000 parameters, P2/4–P3/8–P4/16 high-resolution feature pyramid, NMS-free dual assignment, DFL-free bounding box regression).
- **Performance Target**: mAP@0.5 >= 0.85, minimal Time to First Detection (TTFD), zero false alarm rate.

In [ ]:
# === 1. ENVIRONMENT SETUP & REPOSITORY CLONE ===
import os
import sys
from pathlib import Path
import torch

GITHUB_REPO_URL = "https://github.com/ituvtu/aerial-vehicle-detector.git"
REPO_DIR = Path("/content/aerial-vehicle-detector")

if not REPO_DIR.exists():
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}

# Install dependencies
!pip install -q "ultralytics>=8.4.0" "git+https://github.com/ultralytics/CLIP.git" "pandas" "pyyaml"

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# === 2. VERIFY DATASET & UNPACK IMAGES ===
import yaml
from src.manifest_manager import ensure_images_available, sync_manifest_splits

with open("configs/project_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ensure_images_available(
    manifest_path=cfg["paths"]["manifest_path"],
    images_dir=cfg["paths"]["images_dir"],
    bundle_zip_path=cfg["paths"]["images_bundle_zip"],
    bundle_url=cfg.get("remote_data", {}).get("images_bundle_url"),
)
split_counts = sync_manifest_splits()
print("Frames unpacked. Dataset split distribution:", split_counts)

In [ ]:
# === 3. AUDIT CURATED LABELS & VERIFY RULE 4 COVERAGE ===
!python scripts/03_audit_labels.py

In [ ]:
# === 4. TRAIN YOLO26-PICO-P2 & CALIBRATE VALIDATION THRESHOLD ===
# Architecture: configs/yolo26_pico_p2.yaml (<500k params, P2-P4 strides, Rule 5)
# Calibrates optimal confidence threshold tau* strictly on val split (Rule 1)
!python scripts/04_train.py --epochs 50 --batch 16

In [ ]:
# === 5. EVALUATE ON HELD-OUT TEST SPLIT (Rule 1, Rule 4) ===
# Evaluates on held-out test split (eval: 32179597, eval_B: 13763017)
# Computes mAP@0.5, multi-clip TTFD, false alarms/min, and generates visual artifacts
!python scripts/05_evaluate.py

In [ ]:
# === 6. DISPLAY EVALUATION METRICS & VISUAL RESULTS ===
from IPython.display import Markdown, display, Video
from pathlib import Path

eval_table_path = Path("results/eval_table.md")
if eval_table_path.exists():
    display(Markdown(eval_table_path.read_text(encoding="utf-8")))

demo_mp4_path = Path("results/eval_demo.mp4")
if demo_mp4_path.exists():
    print(f"Generated Evaluation Demo: {demo_mp4_path} ({demo_mp4_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# === 7. PACKAGE DELIVERABLES & DOWNLOAD ===
import datetime
import zipfile
from pathlib import Path

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
deliverables_zip = artifacts_dir / f"eval_deliverables_{timestamp}.zip"

files_to_pack = [
    Path("weights/best.pt"),
    Path("weights/calibration.json"),
    Path("weights/model_arch_report.json"),
    Path("results/eval_metrics.json"),
    Path("results/eval_table.md"),
    Path("results/eval_demo.mp4"),
    Path("data/audit_report.json"),
]

with zipfile.ZipFile(deliverables_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_pack:
        if p.exists():
            zf.write(p, arcname=p.as_posix())
    for crop in Path("results/crops").glob("*.jpg"):
        zf.write(crop, arcname=f"results/crops/{crop.name}")

print(f"Packaged deliverables archive: {deliverables_zip} ({deliverables_zip.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(deliverables_zip))
    print("Download triggered in Colab browser.")
except (ImportError, Exception) as exc:
    print(f"Note: Running locally or download skipped ({exc}). Deliverable saved at {deliverables_zip}")